In [ ]:
import os 

os.chdir('../..')
os.getcwd()

In [ ]:
from ultralytics import YOLO 

MODEL = 'yolo26n.pt'
DATASET = "HomeObjects-3K.yaml"

N_EPOCHS = 40
IMGSIZE = 640

# Check firing

Okay if @@@ INIT for convolutions

In [ ]:
from softstairs_qat import SoftStairsQuantizer, QuantizationConfig


def get_qconfig(strategy, t_start, steps, n_bits=8):
    return QuantizationConfig(
        n_bits=n_bits, normalized=True, t_scheduler_strategy=strategy, t_start=t_start, t_end=1e-4, n_steps=steps
    )

model = YOLO(MODEL)
qconfig = get_qconfig('linear', .1, 2, 4)
quantizer = SoftStairsQuantizer(model, qconfig, excluded_modules={n for n, p in model.named_modules() if 'bn' in n}, verbose=True)

Okay if @@@ FIRED for convolutions

In [ ]:
import torch 

model(torch.randn(1, 3, 640, 640))

Okay of convolutions with `_orig`

In [ ]:
[n for n, p in quantizer.model.named_parameters()]

In [ ]:
assert len(quantizer._hook_handles), 'no handles found'

In [ ]:
print([p[0] for p in model.named_buffers()])

# baseline no-qat

In [ ]:
model = YOLO(MODEL)
results = model.train(
    data=DATASET,
    epochs=N_EPOCHS,
    imgsz=IMGSIZE,
    save_period=10,
    name='no-qat'
)

# Baseline QAT

In [ ]:
from ultralytics.models.yolo.detect import DetectionTrainer
import torch
from torchao.quantization import quantize_, Int8WeightOnlyConfig
from torchao.quantization.qat import QATConfig
from functools import partial

class TorchQATTrainer(DetectionTrainer):
        def __init__(self, *args, qconfig=None, **kwargs):
             super().__init__(*args, **kwargs)
             if qconfig is None:
                  raise ValueError('qconfig not specified')
             self.qconfig = qconfig 

        def get_model(self, cfg=None, weights=None, verbose=True):
            model = super().get_model(cfg, weights, verbose)

            # QAT initialization happens HERE,
            # after the final training model exists.
            quantize_(model, QATConfig(self.qconfig, step="prepare"))

            return model
    

## Int8

In [ ]:


model = YOLO(MODEL)
int8wo = Int8WeightOnlyConfig(group_size=32)


results = model.train(
    data=DATASET,
    epochs=N_EPOCHS,
    imgsz=IMGSIZE,
    name=f'qat-Int8-{N_EPOCHS}e',
    save_period=10, 
    trainer=partial(TorchQATTrainer, qconfig=int8wo)
)


## Int4

In [ ]:
from torchao.quantization import quantize_, Int4WeightOnlyConfig
import torch
from torchao.quantization import quantize_, Int8WeightOnlyConfig
from torchao.quantization.qat import QATConfig


model = YOLO(MODEL)

int4wo = Int4WeightOnlyConfig(group_size=32)



results = model.train(
    data=DATASET,
    epochs=N_EPOCHS,
    imgsz=IMGSIZE,
    name=f'qat-Int4-{N_EPOCHS}e',
    save_period=10,
    trainer=partial(TorchQATTrainer, qconfig=int4wo)
)

# Running

In [ ]:
from itertools import product 

STRATEGIES = ['linear',
               'cos', 'exp', 
            #    'constant', 
               ]
T_START = [.9, 0.5, 0.1]

In [ ]:
import matplotlib.pyplot as plt 
from functools import partial
from softstairs_qat import SoftStairsQuantizer, QuantizationConfig

def get_qconfig(strategy, t_start, steps, n_bits=8):
    return QuantizationConfig(
        n_bits=n_bits, normalized=True, t_scheduler_strategy=strategy, t_start=t_start, t_end=1e-4, n_steps=steps
    )

def run_experiment_nb(model_name, n_epochs, strategy, t_start, n_bits=8):
    model = YOLO(model_name) 
    qconfig = get_qconfig(strategy, t_start, n_epochs, n_bits)
    from ultralytics.models.yolo.detect import DetectionTrainer

    class QATTrainer(DetectionTrainer):

        def get_model(self, cfg=None, weights=None, verbose=True):
            model = super().get_model(cfg, weights, verbose)

            # QAT initialization happens HERE,
            # after the final training model exists.
            self.quantizer = SoftStairsQuantizer(model, qconfig, excluded_modules={n for n, p in model.named_modules() if 'bn' in n})

            return model
    
    model.train(data=DATASET, epochs=n_epochs, name=f'{strategy}-{t_start}-{qconfig.n_bits}b-{n_epochs}e', imgsz=IMGSIZE, trainer=QATTrainer,
    save_period=10 )
    # assert quantizer_storage[0] is not None, 'No Quantizer were Initiated'


In [ ]:
import subprocess
import sys

for i, (t, strat) in enumerate(product(T_START, 
                                       STRATEGIES, 
                                       )):
    try:
        print("Running:", strat, t)

        result = subprocess.run(
            [
                sys.executable,
                "run_training.py",
                "--model", MODEL,
                "--epochs", str(N_EPOCHS),
                "--strategy", strat,
                "--t-start", str(t),
                "--bits", "4",
            ],
            cwd='/home/leostre/Рабочий стол/SoftStairs-QAT',
            check=True,
        )
    except KeyboardInterrupt:
        raise
    except:
        with open('/home/leostre/Рабочий стол/SoftStairs-QAT/experiments/yolo/sdout.txt', 'a') as file:
            print('PASSED', strat, t, file=file)

Overriding model.yaml nc=80 with nc=12

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      6640  ultralytics.nn.modules.block.C3k2            [32, 64, 1, False, 0.25]      
  3                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]                
  4                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  5                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
  6                  -1  1     87040  ultralytics.nn.modules.block.C3k2            [128, 128, 1, True]           
  7                  -1  1    295424  ultralytic

In [ ]:
from itertools import product 


for strat, t in product(
    # ['exp'], [5e-1]
    STRATEGIES, T_START
    ):
    if strat == 'linear' and t == 0.5:
        continue
    print('Running:', strat, t)
    run_experiment_nb(MODEL, n_epochs=2, strategy=strat, t_start=t, n_bits=4)
    